# 🛢️ Sistem Peringatan Dini Gejolak Harga Minyak Goreng di Indonesia
## Menjelang Hari Besar Keagamaan

---

**Program Studi:** Informatika, FMIPA, Universitas Udayana, Jimbaran  
**Tahun:** 2026  
**Anggota Kelompok:**
- Mochammad Riky Hidayat (2408561090)
- I Kadek Candra Gunawan (2408561057)
- I Wayan Gde Adi Suryawirawan (2408561086)

---

### 📋 Ringkasan Proyek

Proyek ini membangun **sistem peringatan dini (early warning system)** berbasis *time series forecasting* untuk mendeteksi potensi lonjakan (gejolak) harga minyak goreng di Indonesia **sebelum** lonjakan tersebut benar-benar terjadi, khususnya menjelang hari besar keagamaan (Ramadan, Idul Fitri, Idul Adha, Nyepi, Natal).

**Model utama:** Facebook/Meta **Prophet** - mendukung pemodelan efek hari libur secara eksplisit, toleran terhadap data hilang, dan sesuai untuk data historis berjumlah terbatas (+-240 titik mingguan).

**Sumber data:** [PIHPS Bank Indonesia / Panel Harga Badan Pangan Nasional](https://www.bi.go.id/hargapangan/TabelHarga/PasarTradisionalDaerah)  
**Rentang data:** 3 Januari 2022 - 24 Agustus 2026, mingguan
github project: https://github.com/candragnawn/spike-oil-price-prediction

---

### 🗂️ Struktur Notebook

| Bagian | Deskripsi |
|--------|-----------|
| 1. Import Library | Memuat semua library yang diperlukan |
| 2. Load Dataset | Membaca data harga dari file Excel |
| 3. Data Cleaning & Preparation | Transformasi, penanganan missing values, format Prophet |
| 4. Load Data Hari Libur | Mendefinisikan kalender hari besar keagamaan |
| 5. Exploratory Data Analysis (EDA) | Statistik deskriptif dan visualisasi |
| 6. Visualisasi Lanjutan | Time series, musiman, korelasi dengan hari libur |

## 1. Import Library

Memuat semua pustaka yang diperlukan untuk analisis data, visualisasi, dan pemodelan.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.core.display_functions import display
import warnings

warnings.filterwarnings('ignore')

# Konfigurasi tampilan
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', 20)

# Palet warna untuk 4 varian minyak goreng
COLORS = {
    'Minyak Goreng': '#E63946',
    'Minyak Goreng Curah': '#F4A261',
    'Minyak Goreng Kemasan Bermerk 1': '#2A9D8F',
    'Minyak Goreng Kemasan Bermerk 2': '#457B9D',
}

print('Library berhasil dimuat!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## 2. Load Dataset

**Sumber:** PIHPS Bank Indonesia - data harga pangan mingguan nasional  
**Format file:** Excel (.xlsx) dengan struktur wide (setiap kolom = satu tanggal)

Dataset yang digunakan:
- `Tabel Harga Minyak Indonesia (2).xlsx` - Data **Nasional** (digunakan sebagai data utama)
- `Tabel Harga Minyak Bali.xlsx` - Data Bali (referensi)

> **Fokus analisis:** Data nasional karena proyek bertujuan untuk early warning di tingkat nasional.

In [ ]:
folder = Path('dataset')
df_raw_all = {f.stem: pd.read_excel(f) for f in folder.glob('*.xlsx')}

print('File yang berhasil dimuat:')
for nama, df_tmp in df_raw_all.items():
    print(f'  - {nama}: {df_tmp.shape[0]} baris x {df_tmp.shape[1]} kolom')

In [ ]:
# Pratinjau data nasional (raw - format wide)
nama_file_nasional = 'Tabel Harga Minyak Indonesia (2)'
df_raw = df_raw_all[nama_file_nasional]

print(f'Pratinjau Data: {nama_file_nasional}')
print(f'Ukuran: {df_raw.shape[0]} baris (komoditas) x {df_raw.shape[1]} kolom')
display(df_raw.head())

## 3. Data Cleaning & Preparation

Langkah-langkah pembersihan dan transformasi data:

1. **Transformasi Wide to Long format**: Mengubah dari format "komoditas sebagai baris, tanggal sebagai kolom" menjadi format tabel panjang yang memudahkan analisis
2. **Konversi tipe data**: 
   - Tanggal: string -> datetime
   - Harga: string dengan pemisah ribuan -> numerik
3. **Penanganan missing values**: Tanda "-" -> NaN; identifikasi lokasi data hilang
4. **Imputasi untuk EDA** (opsional): Interpolasi linear untuk visualisasi saja (bukan untuk pelatihan model)
5. **Pembuatan dataset Prophet**: Format `ds` (tanggal) + `y` (harga) per varian

In [ ]:
# Fungsi: Transformasi wide -> long + pembersihan
def bersihkan_data(df_wide):
    """
    Mengubah dataframe format wide menjadi long,
    membersihkan tanggal dan harga.
    """
    nama_kolom_komoditas = df_wide.columns[1]  # Kolom ke-2 = nama komoditas
    
    # Kolom tanggal = semua kolom selain 'No' dan nama komoditas
    kolom_tanggal = [
        col for col in df_wide.columns
        if col not in ['No', nama_kolom_komoditas, 'Komoditas']
    ]
    
    # Melt: wide -> long
    df_long = pd.melt(
        df_wide,
        id_vars=[col for col in df_wide.columns if col not in kolom_tanggal],
        value_vars=kolom_tanggal,
        var_name='Tanggal_Str',
        value_name='Harga_Raw',
    )
    
    # Standarisasi nama kolom
    df_long = df_long.rename(columns={nama_kolom_komoditas: 'Komoditas'})
    
    # Konversi tanggal: '03/ 01/ 2022' -> datetime
    df_long['Tanggal'] = pd.to_datetime(
        df_long['Tanggal_Str'].astype(str).str.replace(' ', ''),
        format='%d/%m/%Y',
        errors='coerce'
    )
    
    # Konversi harga: '20,000' -> 20000.0, '-' -> NaN
    df_long['Harga'] = pd.to_numeric(
        df_long['Harga_Raw'].astype(str).str.replace(',', '').str.replace('-', ''),
        errors='coerce'
    )
    
    # Kolom tambahan untuk analisis
    df_long['Tahun'] = df_long['Tanggal'].dt.year
    df_long['Bulan'] = df_long['Tanggal'].dt.month
    df_long['Bulan_Nama'] = df_long['Tanggal'].dt.strftime('%B')
    
    # Hapus kolom sementara
    df_long = df_long.drop(columns=['Tanggal_Str', 'Harga_Raw'])
    
    # Urut berdasarkan komoditas & tanggal
    df_long = df_long.sort_values(['Komoditas', 'Tanggal']).reset_index(drop=True)
    
    return df_long


# Terapkan pembersihan
df_long = bersihkan_data(df_raw)

print(f'Data berhasil dibersihkan!')
print(f'Jumlah baris: {len(df_long):,}')
print(f'Komoditas: {df_long["Komoditas"].unique().tolist()}')
print(f'\nPratinjau data (10 baris pertama):')
display(df_long.head(10))

In [ ]:
# Identifikasi & laporan missing values
print('LAPORAN MISSING VALUES\n')
print('-' * 50)

# Tabel missing per komoditas dan tahun
tabel_missing = (
    df_long.groupby(['Komoditas', 'Tahun'])['Harga']
    .apply(lambda x: x.isna().sum())
    .unstack(fill_value=0)
)
print('Jumlah data hilang per komoditas per tahun:')
display(tabel_missing)

# Tanggal spesifik dengan data hilang
missing_detail = df_long[df_long['Harga'].isna()][['Komoditas', 'Tanggal']].drop_duplicates()
if len(missing_detail) > 0:
    print('\nTanggal dengan data hilang:')
    display(missing_detail)
else:
    print('Tidak ada data yang hilang!')

In [ ]:
# Buat dataset format Prophet: ds + y per varian
VARIAN_LIST = [
    'Minyak Goreng',
    'Minyak Goreng Curah',
    'Minyak Goreng Kemasan Bermerk 1',
    'Minyak Goreng Kemasan Bermerk 2',
]

datasets_prophet = {}

for varian in VARIAN_LIST:
    df_varian = (
        df_long[df_long['Komoditas'] == varian]
        [['Tanggal', 'Harga']]
        .rename(columns={'Tanggal': 'ds', 'Harga': 'y'})
        .dropna()   # Hapus baris dengan NaN (Prophet toleran terhadap gap, bukan NaN)
        .sort_values('ds')
        .reset_index(drop=True)
    )
    datasets_prophet[varian] = df_varian

print('Dataset format Prophet berhasil dibuat!\n')
for varian, ds in datasets_prophet.items():
    n_missing_orig = df_long[df_long['Komoditas'] == varian]['Harga'].isna().sum()
    print(f'  {varian}')
    print(f'     Jumlah titik data: {len(ds)} (dari {len(ds) + n_missing_orig} total, {n_missing_orig} dihapus)')
    print(f'     Rentang: {ds["ds"].min().date()} s/d {ds["ds"].max().date()}')
    print()

print('\nContoh dataset Prophet (Minyak Goreng):')
display(datasets_prophet['Minyak Goreng'].head(10))

In [ ]:
# Dataset untuk EDA: imputasi interpolasi linear
# (HANYA untuk visualisasi, BUKAN pelatihan model)
df_eda = df_long.copy()

# Imputasi per komoditas
for varian in VARIAN_LIST:
    mask = df_eda['Komoditas'] == varian
    df_eda.loc[mask, 'Harga'] = (
        df_eda.loc[mask, 'Harga']
        .interpolate(method='linear', limit_direction='both')
    )

print('Dataset EDA (dengan imputasi interpolasi) siap!')
print(f'Sisa missing values: {df_eda["Harga"].isna().sum()}')

## 4. Load Data Hari Libur

Prophet membutuhkan dataframe hari libur dengan kolom:
- `holiday`: nama hari libur
- `ds`: tanggal (datetime)
- `lower_window`: efek dimulai berapa hari **sebelum** hari H
- `upper_window`: efek berlanjut berapa hari **setelah** hari H

**Strategi penentuan window:**
- Hari libur dengan **dampak tinggi terhadap harga pangan** (Idul Fitri, Idul Adha, Imlek, Natal) menggunakan window lebih lebar
- Hari libur **tanpa keterkaitan langsung** dengan harga pangan menggunakan window minimal

Dasar: Permintaan minyak goreng meningkat menjelang hari besar karena kebutuhan memasak meningkat. Efek harga biasanya mulai terasa 1-2 minggu sebelum hari H.

In [ ]:
semua_hari_libur = [
    # 1. TAHUN BARU MASEHI
    {'holiday': 'Tahun Baru', 'ds': '2022-01-01', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Tahun Baru', 'ds': '2023-01-01', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Tahun Baru', 'ds': '2024-01-01', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Tahun Baru', 'ds': '2025-01-01', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Tahun Baru', 'ds': '2026-01-01', 'lower_window': -3, 'upper_window': 1},

    # 2. TAHUN BARU IMLEK
    {'holiday': 'Imlek', 'ds': '2022-02-01', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Imlek', 'ds': '2023-01-22', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Imlek', 'ds': '2024-02-10', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Imlek', 'ds': '2025-01-29', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Imlek', 'ds': '2026-02-17', 'lower_window': -7, 'upper_window': 3},

    # 3. ISRA MIRAJ
    {'holiday': 'Isra Miraj', 'ds': '2022-02-28', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Isra Miraj', 'ds': '2023-02-18', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Isra Miraj', 'ds': '2024-02-08', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Isra Miraj', 'ds': '2025-01-27', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Isra Miraj', 'ds': '2026-01-16', 'lower_window': -1, 'upper_window': 1},

    # 4. NYEPI
    {'holiday': 'Nyepi', 'ds': '2022-03-03', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Nyepi', 'ds': '2023-03-22', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Nyepi', 'ds': '2024-03-11', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Nyepi', 'ds': '2025-03-29', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Nyepi', 'ds': '2026-03-19', 'lower_window': -3, 'upper_window': 1},

    # 5. WAFAT ISA AL MASIH / JUMAT AGUNG
    {'holiday': 'Jumat Agung', 'ds': '2022-04-15', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Jumat Agung', 'ds': '2023-04-07', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Jumat Agung', 'ds': '2024-03-29', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Jumat Agung', 'ds': '2025-04-18', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Jumat Agung', 'ds': '2026-04-03', 'lower_window': -2, 'upper_window': 1},

    # 6. IDUL FITRI (window lebar: efek Ramadan mulai 2 minggu sebelum)
    {'holiday': 'Idul Fitri', 'ds': '2022-05-02', 'lower_window': -14, 'upper_window': 7},
    {'holiday': 'Idul Fitri', 'ds': '2023-04-22', 'lower_window': -14, 'upper_window': 7},
    {'holiday': 'Idul Fitri', 'ds': '2024-04-10', 'lower_window': -14, 'upper_window': 7},
    {'holiday': 'Idul Fitri', 'ds': '2025-03-31', 'lower_window': -14, 'upper_window': 7},
    {'holiday': 'Idul Fitri', 'ds': '2026-03-21', 'lower_window': -14, 'upper_window': 7},

    # 7. HARI BURUH
    {'holiday': 'Hari Buruh', 'ds': '2022-05-01', 'lower_window': 0, 'upper_window': 1},
    {'holiday': 'Hari Buruh', 'ds': '2023-05-01', 'lower_window': 0, 'upper_window': 1},
    {'holiday': 'Hari Buruh', 'ds': '2024-05-01', 'lower_window': 0, 'upper_window': 1},
    {'holiday': 'Hari Buruh', 'ds': '2025-05-01', 'lower_window': 0, 'upper_window': 1},
    {'holiday': 'Hari Buruh', 'ds': '2026-05-01', 'lower_window': 0, 'upper_window': 1},

    # 8. WAISAK
    {'holiday': 'Waisak', 'ds': '2022-05-16', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Waisak', 'ds': '2023-06-04', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Waisak', 'ds': '2024-05-23', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Waisak', 'ds': '2025-05-12', 'lower_window': -3, 'upper_window': 1},
    {'holiday': 'Waisak', 'ds': '2026-05-31', 'lower_window': -3, 'upper_window': 1},

    # 9. KENAIKAN ISA AL MASIH
    {'holiday': 'Kenaikan Isa', 'ds': '2022-05-26', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kenaikan Isa', 'ds': '2023-05-18', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kenaikan Isa', 'ds': '2024-05-09', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kenaikan Isa', 'ds': '2025-05-29', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kenaikan Isa', 'ds': '2026-05-14', 'lower_window': -1, 'upper_window': 1},

    # 10. HARI LAHIR PANCASILA
    {'holiday': 'Pancasila', 'ds': '2022-06-01', 'lower_window': 0, 'upper_window': 0},
    {'holiday': 'Pancasila', 'ds': '2023-06-01', 'lower_window': 0, 'upper_window': 0},
    {'holiday': 'Pancasila', 'ds': '2024-06-01', 'lower_window': 0, 'upper_window': 0},
    {'holiday': 'Pancasila', 'ds': '2025-06-01', 'lower_window': 0, 'upper_window': 0},
    {'holiday': 'Pancasila', 'ds': '2026-06-01', 'lower_window': 0, 'upper_window': 0},

    # 11. IDUL ADHA (window sedang: peningkatan kebutuhan pangan)
    {'holiday': 'Idul Adha', 'ds': '2022-07-09', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Idul Adha', 'ds': '2023-06-29', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Idul Adha', 'ds': '2024-06-17', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Idul Adha', 'ds': '2025-06-06', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Idul Adha', 'ds': '2026-05-27', 'lower_window': -7, 'upper_window': 3},

    # 12. TAHUN BARU ISLAM
    {'holiday': 'Tahun Baru Islam', 'ds': '2022-07-30', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Tahun Baru Islam', 'ds': '2023-07-19', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Tahun Baru Islam', 'ds': '2024-07-07', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Tahun Baru Islam', 'ds': '2025-06-27', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Tahun Baru Islam', 'ds': '2026-06-16', 'lower_window': -1, 'upper_window': 1},

    # 13. KEMERDEKAAN RI
    {'holiday': 'Kemerdekaan RI', 'ds': '2022-08-17', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kemerdekaan RI', 'ds': '2023-08-17', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kemerdekaan RI', 'ds': '2024-08-17', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kemerdekaan RI', 'ds': '2025-08-17', 'lower_window': -1, 'upper_window': 1},
    {'holiday': 'Kemerdekaan RI', 'ds': '2026-08-17', 'lower_window': -1, 'upper_window': 1},

    # 14. MAULID NABI MUHAMMAD SAW
    {'holiday': 'Maulid Nabi', 'ds': '2022-10-08', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Maulid Nabi', 'ds': '2023-09-28', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Maulid Nabi', 'ds': '2024-09-16', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Maulid Nabi', 'ds': '2025-09-05', 'lower_window': -2, 'upper_window': 1},
    {'holiday': 'Maulid Nabi', 'ds': '2026-08-25', 'lower_window': -2, 'upper_window': 1},

    # 15. NATAL (window lebar: peningkatan konsumsi pangan)
    {'holiday': 'Natal', 'ds': '2022-12-25', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Natal', 'ds': '2023-12-25', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Natal', 'ds': '2024-12-25', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Natal', 'ds': '2025-12-25', 'lower_window': -7, 'upper_window': 3},
    {'holiday': 'Natal', 'ds': '2026-12-25', 'lower_window': -7, 'upper_window': 3},

    # 16. PEMILU 2024
    {'holiday': 'Pemilu', 'ds': '2024-02-14', 'lower_window': -1, 'upper_window': 1},
]

holidays_df = pd.DataFrame(semua_hari_libur)
holidays_df['ds'] = pd.to_datetime(holidays_df['ds'])
holidays_df = holidays_df.sort_values('ds').reset_index(drop=True)

print(f'Total {len(holidays_df)} entri hari libur berhasil dimuat!')
print(f'Jumlah jenis hari libur: {holidays_df["holiday"].nunique()}')
print()
print('Ringkasan per jenis hari libur:')
ringkasan_libur = holidays_df.groupby('holiday').agg(
    Jumlah_Tahun=('ds', 'count'),
    Window_Bawah=('lower_window', 'first'),
    Window_Atas=('upper_window', 'first'),
).reset_index()
display(ringkasan_libur)

print('\nContoh 5 baris pertama:')
display(holidays_df.head())

## 5. Exploratory Data Analysis (EDA)

Tujuan EDA:
1. Memahami distribusi statistik harga tiap varian
2. Mengidentifikasi tren jangka panjang
3. Menemukan pola musiman (bulanan)
4. Mendeteksi outlier dan periode gejolak harga
5. Mengeksplorasi korelasi antar varian

In [ ]:
# 5.1 Statistik Deskriptif
print('STATISTIK DESKRIPTIF HARGA MINGGUAN (3 Jan 2022 - 24 Agu 2026)\n')
print('-' * 70)

statistik = (
    df_long.groupby('Komoditas')['Harga']
    .agg(
        N='count',
        Data_Hilang=lambda x: x.isna().sum(),
        Min='min',
        Maks='max',
        Rata_rata='mean',
        Median='median',
        Std_Dev='std',
    )
    .reset_index()
)

# Koefisien variasi (%)
statistik['CV_persen'] = (statistik['Std_Dev'] / statistik['Rata_rata'] * 100).round(2)

# Format pembulatan
for col in ['Min', 'Maks', 'Rata_rata', 'Median', 'Std_Dev']:
    statistik[col] = statistik[col].round(0)

statistik.columns = [
    'Varian Minyak', 'N', 'Data Hilang',
    'Min (Rp)', 'Maks (Rp)', 'Rata-rata (Rp)', 'Median (Rp)',
    'Std. Dev (Rp)', 'CV (%)'
]

display(statistik)

In [ ]:
# 5.2 Grafik Time Series - Semua Varian

# Hari libur utama untuk ditandai
target_holidays = ['Idul Fitri', 'Idul Adha', 'Nyepi', 'Natal', 'Imlek']
warna_libur = {'Idul Fitri': '#e76f51', 'Idul Adha': '#f4a261', 'Nyepi': '#2a9d8f', 'Natal': '#e9c46a', 'Imlek': '#457b9d'}

fig, axes = plt.subplots(4, 1, figsize=(16, 20), sharex=True)
fig.suptitle(
    'Tren Harga Minyak Goreng Mingguan Nasional (3 Januari 2022 - 24 Agustus 2026)',
    fontsize=16, fontweight='bold', y=0.98
)

for ax, varian in zip(axes, VARIAN_LIST):
    data_v = df_eda[df_eda['Komoditas'] == varian].sort_values('Tanggal')
    
    # Plot harga
    ax.plot(data_v['Tanggal'], data_v['Harga'],
            color=COLORS[varian], linewidth=2, label=varian)
    ax.fill_between(data_v['Tanggal'], data_v['Harga'],
                    alpha=0.15, color=COLORS[varian])
    
    # Tandai hari libur utama (garis vertikal)
    for holiday in target_holidays:
        hol_data = holidays_df[holidays_df['holiday'] == holiday]
        for tgl in hol_data['ds']:
            if data_v['Tanggal'].min() <= tgl <= data_v['Tanggal'].max():
                ax.axvline(x=tgl, color=warna_libur[holiday], alpha=0.5, linewidth=1.5, linestyle='--')
    
    ax.set_title(varian, fontsize=12, pad=8)
    ax.set_ylabel('Harga (Rp)', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

# Tambahkan legenda hari libur
legend_items = [mpatches.Patch(color=warna_libur[h], alpha=0.7, label=h) for h in target_holidays]
fig.legend(handles=legend_items, loc='lower center',
           ncol=5, fontsize=10, title='Hari Besar Keagamaan',
           bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.show()

In [ ]:
# 5.3 Grafik Time Series - Semua Varian dalam 1 Panel

fig, ax = plt.subplots(figsize=(16, 7))

for varian in VARIAN_LIST:
    data_v = df_eda[df_eda['Komoditas'] == varian].sort_values('Tanggal')
    ax.plot(data_v['Tanggal'], data_v['Harga'],
            label=varian, color=COLORS[varian], linewidth=2)

# Shading periode hari libur utama
for holiday in target_holidays:
    hol_data = holidays_df[holidays_df['holiday'] == holiday]
    for _, row in hol_data.iterrows():
        mulai = row['ds'] + pd.Timedelta(days=row['lower_window'])
        akhir = row['ds'] + pd.Timedelta(days=row['upper_window'])
        ax.axvspan(mulai, akhir, alpha=0.1, color=warna_libur[holiday], label='_nolegend_')

ax.set_title('Perbandingan Harga 4 Varian Minyak Goreng Nasional (Area diarsir = Periode Hari Libur)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Harga (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Distribusi Harga per Bulan (Boxplot Musiman)
urutan_bulan = ['January','February','March','April','May','June',
                'July','August','September','October','November','December']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Distribusi Harga per Bulan - Pola Musiman 4 Varian Minyak Goreng',
             fontsize=15, fontweight='bold', y=1.01)

for ax, varian in zip(axes.flat, VARIAN_LIST):
    data_v = df_eda[df_eda['Komoditas'] == varian].dropna(subset=['Harga'])
    
    # Filter hanya bulan yang ada di data
    bulan_ada = [b for b in urutan_bulan if b in data_v['Bulan_Nama'].values]
    
    sns.boxplot(
        data=data_v,
        x='Bulan_Nama',
        y='Harga',
        order=bulan_ada,
        color=COLORS[varian],
        ax=ax,
        linewidth=1.2,
        fliersize=3,
    )
    
    ax.set_title(varian, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Harga (Rp)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Matriks Rata-rata Harga Bulanan (Heatmap)
pivot_bulanan = (
    df_eda.groupby(['Bulan', 'Bulan_Nama', 'Komoditas'])['Harga']
    .mean()
    .unstack('Komoditas')
    .reset_index()
    .sort_values('Bulan')
)

# Format untuk heatmap
heatmap_data = pivot_bulanan.set_index('Bulan_Nama')[VARIAN_LIST]

# Reorder bulan
urutan = [b for b in urutan_bulan if b in heatmap_data.index]
heatmap_data = heatmap_data.loc[urutan]

fig, ax = plt.subplots(figsize=(14, 6))

# Label kolom disingkat
label_kolom = ['MG', 'MG Curah', 'MG Kemasan 1', 'MG Kemasan 2']
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    xticklabels=label_kolom,
    cbar_kws={'label': 'Rata-rata Harga (Rp)'},
)

ax.set_title('Matriks Rata-rata Harga Bulanan per Varian Minyak Goreng',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Varian Minyak Goreng')
ax.set_ylabel('Bulan')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# 5.6 Pola Missing Values
print('POLA DATA HILANG PER TAHUN\n')

tabel_missing_tahun = (
    df_long.groupby(['Komoditas', 'Tahun'])['Harga']
    .apply(lambda x: x.isna().sum())
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    tabel_missing_tahun,
    annot=True,
    fmt='d',
    cmap='Reds',
    linewidths=0.5,
    ax=ax,
    vmin=0, vmax=5,
)
ax.set_title('Jumlah Data Hilang per Komoditas per Tahun', fontsize=12, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('Tabel lengkap:')
display(tabel_missing_tahun)

## 6. Visualisasi Lanjutan

Analisis lebih mendalam:
- Tren per tahun
- Korelasi antar varian
- Perubahan harga mingguan (volatilitas)
- Perbandingan harga menjelang hari besar keagamaan

In [ ]:
# 6.1 Rata-rata Harga Tahunan
rata_tahunan = (
    df_eda.groupby(['Tahun', 'Komoditas'])['Harga']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))

for varian in VARIAN_LIST:
    data_v = rata_tahunan[rata_tahunan['Komoditas'] == varian]
    ax.plot(data_v['Tahun'], data_v['Harga'],
            marker='o', markersize=8,
            label=varian, color=COLORS[varian], linewidth=2.5)
    # Annotasi nilai
    for _, row in data_v.iterrows():
        ax.annotate(f'Rp{row["Harga"]:,.0f}',
                    xy=(row['Tahun'], row['Harga']),
                    xytext=(0, 12), textcoords='offset points',
                    ha='center', fontsize=7.5, color=COLORS[varian])

ax.set_title('Tren Rata-rata Harga Minyak Goreng Tahunan',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('Rata-rata Harga (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xticks(rata_tahunan['Tahun'].unique())

plt.tight_layout()
plt.show()

In [ ]:
# 6.2 Volatilitas Harga Mingguan (Perubahan %)
fig, axes = plt.subplots(4, 1, figsize=(16, 18), sharex=True)
fig.suptitle('Perubahan Harga Mingguan (%) - Analisis Volatilitas',
             fontsize=14, fontweight='bold', y=1.0)

for ax, varian in zip(axes, VARIAN_LIST):
    data_v = df_eda[df_eda['Komoditas'] == varian].sort_values('Tanggal').copy()
    data_v['Perubahan_Pct'] = data_v['Harga'].pct_change() * 100
    
    # Bar chart: merah = naik, hijau = turun
    warna = ['#e63946' if v > 0 else '#2a9d8f' for v in data_v['Perubahan_Pct'].fillna(0)]
    ax.bar(data_v['Tanggal'], data_v['Perubahan_Pct'], color=warna, width=5, alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axhline(y=2, color='orange', linewidth=1, linestyle='--', alpha=0.6, label='+-2% threshold')
    ax.axhline(y=-2, color='orange', linewidth=1, linestyle='--', alpha=0.6)
    
    # Highlight periode hari libur
    for holiday in target_holidays:
        hol_data = holidays_df[holidays_df['holiday'] == holiday]
        for _, row in hol_data.iterrows():
            mulai = row['ds'] + pd.Timedelta(days=row['lower_window'])
            akhir = row['ds'] + pd.Timedelta(days=row['upper_window'])
            ax.axvspan(mulai, akhir, alpha=0.1, color='gray')
    
    ax.set_title(varian, fontsize=11)
    ax.set_ylabel('Perubahan (%)')
    ax.grid(axis='y', alpha=0.3)
    
    # Statistik perubahan
    max_naik = data_v['Perubahan_Pct'].max()
    max_turun = data_v['Perubahan_Pct'].min()
    ax.text(0.01, 0.95, f'Max naik: +{max_naik:.1f}% | Max turun: {max_turun:.1f}%',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Korelasi Antar Varian Minyak Goreng

# Pivot: tanggal x varian
df_pivot = df_eda.pivot_table(index='Tanggal', columns='Komoditas', values='Harga')
df_pivot.columns.name = None

# Matriks korelasi
corr_matrix = df_pivot.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel kiri: Heatmap korelasi
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    mask=mask,
    linewidths=1,
    ax=axes[0],
    square=True,
)
axes[0].set_title('Matriks Korelasi Harga (Pearson)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].tick_params(axis='y', rotation=0)

# Singkat nama
label_singkat = {'Minyak Goreng': 'MG', 'Minyak Goreng Curah': 'Curah',
                 'Minyak Goreng Kemasan Bermerk 1': 'Kemasan 1',
                 'Minyak Goreng Kemasan Bermerk 2': 'Kemasan 2'}
axes[0].set_xticklabels([label_singkat.get(t.get_text(), t.get_text())
                         for t in axes[0].get_xticklabels()])
axes[0].set_yticklabels([label_singkat.get(t.get_text(), t.get_text())
                         for t in axes[0].get_yticklabels()])

# Panel kanan: Scatter plot MG vs Curah
axes[1].scatter(
    df_pivot['Minyak Goreng'],
    df_pivot['Minyak Goreng Curah'],
    alpha=0.5, s=30, color='#457b9d',
    label=f"r = {corr_matrix.loc['Minyak Goreng', 'Minyak Goreng Curah']:.3f}"
)
# Garis regresi
z = np.polyfit(
    df_pivot['Minyak Goreng'].dropna(),
    df_pivot['Minyak Goreng Curah'].dropna(),
    1
)
p = np.poly1d(z)
x_line = np.linspace(df_pivot['Minyak Goreng'].min(), df_pivot['Minyak Goreng'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Regresi Linear')

axes[1].set_title('Scatter Plot: MG vs MG Curah', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Harga MG (Rp)')
axes[1].set_ylabel('Harga MG Curah (Rp)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Matriks korelasi lengkap:')
display(corr_matrix.round(4))

In [ ]:
# 6.4 Efek Hari Besar Keagamaan terhadap Harga
# Fokus: Analisis sekitar hari besar keagamaan (+-4 minggu)

target_holidays = ['Idul Fitri', 'Idul Adha', 'Nyepi', 'Natal', 'Imlek']

fig, axes = plt.subplots(len(target_holidays), 1, figsize=(14, 6 * len(target_holidays)))

# Fokus pada MG (Minyak Goreng umum) sebagai representasi
data_mg = df_eda[df_eda['Komoditas'] == 'Minyak Goreng'].sort_values('Tanggal').copy()
WINDOW = 28  # +-4 minggu
palette_tahun = {2022: '#e63946', 2023: '#457b9d', 2024: '#2a9d8f', 2025: '#e9c46a', 2026: '#f4a261'}

for i, holiday_name in enumerate(target_holidays):
    ax = axes[i]
    holiday_data = holidays_df[holidays_df['holiday'] == holiday_name]
    
    for _, row in holiday_data.iterrows():
        tgl_h = row['ds']
        tahun = tgl_h.year
        mulai = tgl_h - pd.Timedelta(days=WINDOW)
        akhir = tgl_h + pd.Timedelta(days=WINDOW)
        
        df_window = data_mg[(data_mg['Tanggal'] >= mulai) & (data_mg['Tanggal'] <= akhir)].copy()
        if len(df_window) == 0:
            continue
        
        # Hitung hari relatif dari hari besar
        df_window['Hari_Relatif'] = (df_window['Tanggal'] - tgl_h).dt.days
        
        ax.plot(
            df_window['Hari_Relatif'],
            df_window['Harga'],
            label=f'{tahun} ({tgl_h.strftime("%d %b")})',
            color=palette_tahun.get(tahun, '#000000'),
            linewidth=2.5,
            marker='o', markersize=5,
        )

    ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='H-0')
    ax.axvspan(-14, 0, alpha=0.08, color='orange', label='2 Minggu Sebelum')

    ax.set_title(f'Pola Harga Minyak Goreng Menjelang & Setelah {holiday_name} (Hari ke-0 = Hari H)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(f'Hari Relatif terhadap {holiday_name}')
    ax.set_ylabel('Harga (Rp)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
    ax.legend(fontsize=10, loc='upper right', ncol=2)
    ax.grid(alpha=0.3)
    ax.set_xticks(range(-28, 29, 7))
    ax.set_xticklabels([f'H{x:+d}' if x != 0 else 'H-0' for x in range(-28, 29, 7)])

plt.tight_layout()
plt.show()
